# Module 1 -- DRG-Net replication runs (classification + segmentation)

Re-runs DRG-Net's own training code (pinned commit `0da1bbe`) the way **they** used each dataset,
so our Module 1 numbers can be compared with theirs. DRG-Net trains **one model per dataset**
(never pooled), each with its own config:

| Task | Dataset | DRG-Net script + config | Split | Epochs |
|---|---|---|---|---|
| Grading | FGADR (1,842) | `main.py` + `fgadr.yaml` (SGD, weighted CE) | rows 0-1499 train, 1500-1841 test, **val = test** (their released `fgadr_data_dict.pkl`) | 20 |
| Grading | IDRiD (413 / 103) | `main.py` + `idrid.yaml` (Adam, focal loss, progressively balanced) | official train / test, **val = test** | 60 |
| Grading | EyePACS | `main.py` + `eyepacs_ce.yaml` (SGD, plain CE) | not released -- we use the Kaggle split (train / public test / private test) | 25 |
| Lesion seg. EX/HE/MA/SE | FGADR (401 with all 4 masks) | `train_fgadr.py` + `config_fgadr.py`, `--seed 765 --preprocess 2` | 60 / 20 / 20 by row order of `DR_Seg_Grading_Label_Filtered.csv` | 1500 (default here: **100**) |
| Lesion seg. EX/HE/MA/SE | IDRiD (54 / 27) | `train.py` + `config.py`, `--seed 765 --preprocess 2` | 80 / 20 of training set; test = official test set | 2500 (default here: **100**) |
| Lesion seg. EX/HE/MA/SE | **DDR** (383 / 149 / 225) -- *ours, not in DRG-Net* | `train.py` + `config.py` (same as IDRiD), `--seed 765 --preprocess 2` | official train / val (model selection) / test | -- (default here: **100**) |
| Lesion seg. EX/HE/MA/SE | **FGADR + DDR combined** -- *ours; the pipeline's segmenters* | `train.py` + `config.py`, `--seed 765 --preprocess 2` | FGADR's 60/20 + DDR's train/val pooled; scored on each test set separately | -- (default here: **200**, per-lesion override) |

EyePACS has **no lesion masks**, so DRG-Net never trains segmentation on it (their
`eyepacs_dataset.py` only loads images + FOV masks, unused by any training script).

**DDR** (Li et al., 2019) is our addition: DRG-Net never used it, so its numbers are extra
results with DRG-Net's IDRiD training recipe, not a replication. Unlike FGADR/IDRiD it has a real
validation split, so the best epoch is chosen on val and the test set is only used for the final
score. The DDR-only models are reported only.

**FGADR + DDR combined** (also ours): one model per lesion trained on both datasets' training
images (~623) and selected on both validation sets (~230). These are the lesion segmenters the
pipeline uses by default (`MODULE1_SOURCE = 'drgnet_combined'` in notebooks 04_05-10). The
evaluation cell scores every model (FGADR-only, IDRiD, DDR-only, combined) on every test set, so
the thesis can show whether pooling helped -- IDRiD's test set, which neither FGADR nor DDR
trains on, is the fairest check. (DRG-Net's `--preprocess 2` "contrast enhancement" discards
its CLAHE result -- `clahe.apply()` is never assigned -- so it is effectively the original image,
matching the FGADR runs and what the pipeline feeds the models.)

**Known gaps vs the paper -- state these when comparing numbers:**
- The paper's headline segmentation results (e.g. FGADR "S-Net + PD-Net + TATL", the table in
  `module1/README.md`) use their TATL transfer-learning setup, which is **not in the released
  code**. These runs reproduce the released `NO_TATL` training, so expect lower numbers than the
  TATL row and compare against the paper's non-TATL baseline where it reports one.
- `val == test` for FGADR/IDRiD grading is DRG-Net's own protocol (checkpoint selection sees the
  test set). Kept for replication.
- **Epochs:** the comparison segmentation runs default to **100 epochs** and the pipeline's FGADR + DDR
  combined segmenters to **200** (`SEG_EPOCHS`, per-lesion override, e.g. MA 300) instead of DRG-Net's
  1500 (FGADR) / 2500 (IDRiD) -- 200 epochs on ~623 pooled images is ~125k image passes, ~35 % of
  DRG-Net's ~360k (1,500 epochs x 240 FGADR images); 1,500 would be ~230 h on Colab. Report it as
  "DRG-Net's training protocol at a reduced epoch budget", not as a full replication. Training
  steps differ by dataset size: FGADR ~240 training images (~60 steps per epoch, ~6k steps),
  IDRiD ~43 (~11, ~1.1k -- the most under-trained), DDR 383 (~96, ~10k), combined ~623 (~156,
  ~16k). The LR stays at 1e-3 (DRG-Net's StepLR only steps every 200 epochs). Check
  `logs/*ap_during_learning*.txt`: if AP is still rising at the last evaluation, raise that run's
  `SEG_EPOCHS` and re-run -- it continues from epoch 100. Classification keeps DRG-Net's own
  20 / 60 / 25 (`CLS_EPOCHS` overrides them).
- **Checkpoint / resume:** `module1/drgnet_resume_patch.py` adds resume to DRG-Net's training
  scripts (they can't resume on their own -- see that file). Checkpoints are written straight to
  Drive; if Colab disconnects, re-run the notebook and every run continues from its last saved
  epoch (classification: every epoch; segmentation: every `SEG_SAVE_EVERY` epochs). Tested:
  an interrupted + resumed classification run gives bit-identical weights to an uninterrupted
  one; segmentation resumes the epoch count, LR schedule and best-AP tracking (its data shuffling
  differs after a resume, so it is close but not bit-identical).
- Modern torch/torchvision/Pillow instead of their 2022 pins (same compatibility patches as
  notebooks 02/03), so expect small run-to-run variance.

Everything is written to `Thesis_Datasets/module1_runs/drgnet_replication_prefinals/` on Drive
(`RUN_SUFFIX`), separate from the POC runs of notebooks 02/03 and from the earlier
`drgnet_replication/` runs, which stay untouched for before/after comparison. A run with nothing in
the new folder yet starts from its copy in `drgnet_replication/` (so the 100-epoch segmenters continue
to 200 instead of starting over).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('\u26a0 Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os, glob, json
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'AllModuleEdits-PreFinals'  # <-- pre-finals fixes (CODE_FIXES_AllModuleEdits-PreFinals.md); 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache')
MODULE2_RUNS_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs')
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
os.makedirs(MODULE2_RUNS_DIR, exist_ok=True)

# This branch's runs go to drgnet_replication<RUN_SUFFIX>/ so the mod-2-NonLongIntegration models and
# results stay untouched. A run with nothing there yet starts from its copy in PREV_REPL_DIR (e.g. the
# 100-epoch segmenters continue to SEG_EPOCHS instead of starting over); '' = the earlier folder.
RUN_SUFFIX = '_prefinals'
PREV_REPL_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_runs', 'drgnet_replication')
REPL_DIR = PREV_REPL_DIR + RUN_SUFFIX
REPL_PKL_DIR = os.path.join(REPL_DIR, 'pkl')
for _d in ('pkl', 'classification', 'segmentation', 'logs'):
    os.makedirs(os.path.join(REPL_DIR, _d), exist_ok=True)

# Which runs to do this session. Default = only what the pipeline (notebooks 04_05-10) needs:
# the FGADR classifier + the FGADR + DDR combined lesion segmenters (~6-7 h on a T4). The others
# are comparison results for the thesis (FGADR-only / IDRiD / DDR-only models): set them True in
# a later session -- finished runs are kept, so nothing is retrained.
RUN = {
    'fgadr_cls': True,      # PIPELINE: the grader
    'comb_seg': True,       # PIPELINE: FGADR + DDR combined lesion segmenters (needs FGADR + a DDR zip)
    'idrid_cls': False,     # comparison: IDRiD grader (~30 min)
    'eyepacs_cls': False,   # optional: EyePACS grader (pickle fixed on this branch) -- needs the EyePACS zip, several hours
    'fgadr_seg': False,     # comparison: FGADR-only lesion segmenters
    'idrid_seg': False,     # comparison: IDRiD lesion segmenters (needs IDRiD's "A. Segmentation" in archive.zip)
    'ddr_seg': False,       # comparison: DDR-only lesion segmenters (needs a DDR zip on Drive)
}
SEG_LESIONS = ['EX', 'HE', 'MA', 'SE']
# Total epochs per run. An int, or {'default': n, '<lesion>': m} for per-lesion targets. DRG-Net:
# 1500 (FGADR) / 2500 (IDRiD). The pipeline's FGADR + DDR segmenters train to 200 (~125k image
# passes on ~623 images, ~35 % of DRG-Net's ~360k on FGADR); runs resume from their last checkpoint,
# so 100 -> 200 continues the earlier runs (~3.8 h per lesion on a T4, ~15 h for all four). MA was
# still improving at epoch 100 (eval ROC-AUC 0.856 -> 0.973): give it e.g. 300 if compute allows.
# The best epoch (eval AP) is still the one kept as model_2.pth.tar.
SEG_EPOCHS = {'fgadr': 100, 'idrid': 100, 'ddr': 100, 'comb': {'default': 200, 'MA': 200}}

# Small-lesion loss (item 1.4): weight of a soft Dice term added to DRG-Net's cross-entropy
# (DRG_SEG_DICE_WEIGHT, module1/drgnet_resume_patch.py), per run and lesion. On for the pipeline's
# microaneurysm model only -- MA has the lowest test AP (0.204 vs DRG-Net's 0.417) and its few pixels
# barely move the CE loss. {} = DRG-Net's loss everywhere (the comparison runs never get it).
SEG_DICE_WEIGHT = {'comb': {'MA': 0.5}}

def seg_epochs(dataset, lesion):
    e = SEG_EPOCHS[dataset]
    return e.get(lesion, e['default']) if isinstance(e, dict) else e
SEG_EVAL_EVERY = {'fgadr': 20, 'idrid': 20, 'ddr': 20, 'comb': 10}   # 5-10 evaluations (best-epoch picks) per run
DDR_SIZE = 1280       # DDR images become DDR_SIZE x DDR_SIZE (FGADR's resolution); 1536 on High-RAM
SEG_SAVE_EVERY = 5                              # resume checkpoint (model_last.pth.tar) interval
CLS_EPOCHS = {'fgadr': None, 'idrid': None, 'eyepacs': None}  # None = DRG-Net's 20 / 60 / 25
# EyePACS is ~73 % grade 0. DRG-Net's eyepacs_ce.yaml samples instance_balanced; the pipeline's grader
# uses progressively_balanced (class-balanced -> instance-balanced over training). None = DRG-Net's.
CLS_SAMPLING = {'eyepacs': 'progressively_balanced'}
RESUME = True     # continue every run from its last checkpoint on Drive (False = start over)
# Checkpoints are written to the VM's own disk and copied to Drive every N epochs and at the end of
# each run. Writing them straight to Drive every few epochs leaves an old copy each time (Trash /
# old versions), which fills a 15 GB Drive within hours -- after that, model files silently fail
# to save. A disconnect loses at most N-1 epochs; the run resumes from the last Drive copy.
SEG_MIRROR_EVERY = 25   # segmentation (a multiple of SEG_SAVE_EVERY)
CLS_MIRROR_EVERY = 5    # classification
LOCAL_RUNS = '/content/drg_local'

In [ ]:
# Unzip datasets locally (same conventions as notebooks 01-03 and 09).
import shutil
os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FGADR-Seg-set_Release.zip"
!unzip -q -n "$DRIVE_DATA_DIR/archive.zip" -d IDRiD_dataset_root
_c = glob.glob('/content/data/IDRiD_dataset_root/**/IDRiD_dataset', recursive=True)
if _c and not os.path.isdir('/content/data/IDRiD_dataset'):
    shutil.move(_c[0], '/content/data/IDRiD_dataset')
IDRID_ROOT = '/content/data/IDRiD_dataset' if os.path.isdir('/content/data/IDRiD_dataset') else '/content/data/IDRiD_dataset_root'
print('IDRiD root:', IDRID_ROOT, os.listdir(IDRID_ROOT))

_seg = glob.glob('/content/data/**/Seg-set', recursive=True)
assert _seg, 'No FGADR Seg-set folder found -- check FGADR-Seg-set_Release.zip'
FGADR_ROOT = '/content/data/FGADR-Seg-set_Release/Seg-set'
if _seg[0] != FGADR_ROOT and not os.path.exists(FGADR_ROOT):
    os.makedirs(os.path.dirname(FGADR_ROOT), exist_ok=True); os.symlink(_seg[0], FGADR_ROOT)

DDR_ROOT = '/content/data/DDR_raw'
if RUN['ddr_seg'] or RUN['comb_seg']:
    _z = sorted(set(glob.glob(os.path.join(DRIVE_DATA_DIR, '*DDR*.zip')) + glob.glob(os.path.join(DRIVE_DATA_DIR, '*ddr*.zip'))
                    + glob.glob(os.path.join(DRIVE_DATA_DIR, '*DRR*.zip'))))
    if _z:
        print('DDR zip:', _z[0])
        !unzip -q -n "{_z[0]}" -d "{DDR_ROOT}"
    else:
        print('No DDR zip on Drive (a .zip with DDR in its name) -- DDR and combined segmentation skipped')
        RUN['ddr_seg'] = RUN['comb_seg'] = False

if RUN['eyepacs_cls']:
    _z = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, '*eyepacs*.zip')) + glob.glob(os.path.join(DRIVE_DATA_DIR, '*EyePACS*.zip')))
    EYEPACS_DIR = '/content/data/eyepacs'
    if _z:
        !unzip -q -n "{_z[0]}" -d "{EYEPACS_DIR}"
    else:
        print('No EyePACS zip on Drive (see notebook 09) -- EyePACS grader skipped; the pipeline uses the FGADR grader')
        RUN['eyepacs_cls'] = False

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}
REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"
%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch matplotlib scipy torchmetrics torch-fidelity

# DRG-Net reference implementation + the same compatibility patches as notebooks 02/03.
%cd /content
if not os.path.isdir('/content/dr-joint-learning'):
    !git clone https://github.com/DFKI-Interactive-Machine-Learning/dr-joint-learning.git dr-joint-learning
    !cd dr-joint-learning && git checkout 0da1bbe885f438390a0e94ec486283d9b21d5547
DRG = '/content/dr-joint-learning'
!sed -i 's/fillcolor=aug_args.value_fill/fill=aug_args.value_fill/' {DRG}/dr_classification/data/transforms.py
_f = f'{DRG}/dr_segmentation/transform/functional.py'
_t = open(_f).read().replace('from PIL import Image, ImageOps, ImageEnhance, PILLOW_VERSION',
                             "from PIL import Image, ImageOps, ImageEnhance\nPILLOW_VERSION = '9.0.0'")
open(_f, 'w').write(_t)
!pip install -q munch self-attention-cv pretrainedmodels cnn_finetune ipdb tensorboard
# Checkpoint/resume support (DRG-Net's scripts can't resume on their own). Idempotent.
!python {REPO_CODE_DIR}/module1/drgnet_resume_patch.py {DRG}
%cd {REPO_CODE_DIR}

## Classification (DRG-Net `dr_classification/main.py`)

Each run prints `test acc` / `quadratic kappa` twice at the end -- for
`best_validation_weights.pt` and for `final_weights.pt` (DRG-Net's own `main.py`). Logs are
tee'd to Drive and summarised at the bottom of this notebook.

In [ ]:
import re

def restore_dir(drive_dir, local_dir):
    """New runtime: copy a run's Drive files back to the VM disk so training resumes from them.
    A run not yet in REPL_DIR is restored from the same run in PREV_REPL_DIR (read only)."""
    os.makedirs(local_dir, exist_ok=True)
    if not (os.path.isdir(drive_dir) and os.listdir(drive_dir)) and RUN_SUFFIX and drive_dir.startswith(REPL_DIR + os.sep):
        _prev = PREV_REPL_DIR + drive_dir[len(REPL_DIR):]
        if os.path.isdir(_prev):
            print(f'  {os.path.relpath(drive_dir, REPL_DIR)}: starting from the earlier run in {_prev}')
            drive_dir = _prev
    if os.path.isdir(drive_dir):
        for f in os.listdir(drive_dir):
            src, dst = os.path.join(drive_dir, f), os.path.join(local_dir, f)
            if os.path.isfile(src) and not os.path.isfile(dst):
                shutil.copy2(src, dst)

def sync_dir(local_dir, drive_dir):
    """Copy files changed since the last copy (VM disk -> Drive); epoch_N.pt snapshots stay local."""
    os.makedirs(drive_dir, exist_ok=True)
    for f in os.listdir(local_dir):
        src, dst = os.path.join(local_dir, f), os.path.join(drive_dir, f)
        if (os.path.isfile(src) and not f.endswith('.tmp') and not f.startswith('epoch_')
                and (not os.path.isfile(dst) or os.path.getmtime(src) > os.path.getmtime(dst) + 1)):
            shutil.copy2(src, dst)

def check_on_drive(drive_dir, names, what, local_dir=None):
    """Stop loudly if a run's model files are not on Drive (instead of finding out hours later):
    either training never produced them (it crashed -- see its output above) or they were
    produced on this runtime but could not be copied to Drive (Drive full)."""
    missing = [n for n in names if not os.path.isfile(os.path.join(drive_dir, n))
               or os.path.getsize(os.path.join(drive_dir, n)) == 0]
    if not missing:
        print(f'  ✓ {what}: ' + ', '.join(names) + ' saved to Drive')
        return
    if local_dir and not all(os.path.isfile(os.path.join(local_dir, n)) for n in missing):
        raise RuntimeError(f'{what}: training did not produce {missing} -- it stopped early; see the '
                           'error messages in the output above.')
    raise RuntimeError(f'{what}: {missing} not saved to Drive ({drive_dir}). Most likely your Google '
                       'Drive is full: check drive.google.com/settings/storage, empty the Trash, then '
                       're-run this cell (training resumes from the copy on this runtime).')
    print(f'  ✓ {what}: ' + ', '.join(names) + ' saved to Drive')

def cls_dirs(name):
    """(VM-disk save folder DRG-Net writes to, Drive folder it is copied to)."""
    return (os.path.join(LOCAL_RUNS, 'classification', name, 'saves'),
            os.path.join(REPL_DIR, 'classification', name, 'saves'))

def make_config(src_yaml, name, pkl):
    """Copy one of our DRG-Net config copies with data_index pointed at our pickle, save_path/log_path
    at the VM disk (copied to Drive by train_cls) and epochs if CLS_EPOCHS overrides them."""
    s = open(os.path.join(REPO_CODE_DIR, 'module1', 'configs', src_yaml)).read()
    if CLS_EPOCHS.get(name):
        s = re.sub(r"(?m)^(\s+)epochs: \d+", rf"\g<1>epochs: {CLS_EPOCHS[name]}", s, count=1)
    if CLS_SAMPLING.get(name):
        s = re.sub(r"sampling_strategy: \w+", f"sampling_strategy: {CLS_SAMPLING[name]}", s, count=1)
    s = re.sub(r"data_index: .*", f"data_index: '{pkl}'", s, count=1)
    s = re.sub(r"save_path: .*", f"save_path: '{cls_dirs(name)[0]}'", s, count=1)
    s = re.sub(r"log_path: .*", f"log_path: '{LOCAL_RUNS}/classification/{name}/logs'", s, count=1)
    os.makedirs(cls_dirs(name)[0], exist_ok=True)
    os.makedirs(f'{LOCAL_RUNS}/classification/{name}/logs', exist_ok=True)
    dst = os.path.join(DRG, 'dr_classification', 'configs', f'{name}_drgnet_repl.yaml')
    open(dst, 'w').write(s)
    return f'configs/{name}_drgnet_repl.yaml'

FGADR_PKL = os.path.join(REPL_PKL_DIR, 'fgadr_release_split.pkl')
IDRID_PKL = os.path.join(REPL_PKL_DIR, 'idrid_official_split.pkl')
# _v2: built with the fixed Usage handling (blank = original training images -> train)
EYEPACS_PKL = os.path.join(REPL_PKL_DIR, 'eyepacs_official_split_v2.pkl')
# DRG_CLS_RESUME=1 -> train() continues from <save_path>/resume_state.pt if it exists.
CLS_ENV = 'DRG_CLS_RESUME=1' if RESUME else 'DRG_CLS_RESUME=0'

def train_cls(name, src_yaml, pkl):
    """One DRG-Net classification run: restore its Drive saves to the VM disk, train (copying to
    Drive every CLS_MIRROR_EVERY epochs), copy the final files to Drive and check they arrived."""
    cfg = make_config(src_yaml, name, pkl)
    local_dir, drive_dir = cls_dirs(name)
    if RESUME:
        restore_dir(drive_dir, local_dir)
    log = os.path.join(REPL_DIR, 'logs', f'cls_{name}.log')
    !cd "{DRG}/dr_classification" && {CLS_ENV} DRG_CLS_MIRROR_DIR="{drive_dir}" DRG_CLS_MIRROR_EVERY={CLS_MIRROR_EVERY} python main.py -c {cfg} -overwrite 2>&1 | tee -a "{log}"
    sync_dir(local_dir, drive_dir)
    check_on_drive(drive_dir, ['best_validation_weights.pt', 'final_weights.pt'], f'{name} classifier', local_dir)

%cd {REPO_CODE_DIR}
# FGADR: DRG-Net's released split, verified file-by-file against their fgadr_data_dict.pkl.
!python module1/dataprep/make_fgadr_classification_pkl.py --fgadr-root "{FGADR_ROOT}" --out "{FGADR_PKL}" --split release
_idrid_grading = glob.glob(f'{IDRID_ROOT}/**/B. Disease Grading', recursive=True)
assert _idrid_grading, f"No 'B. Disease Grading' under {IDRID_ROOT}"
!python module1/dataprep/make_idrid_classification_pkl.py --idrid-root "{_idrid_grading[0]}" --out "{IDRID_PKL}"
if RUN['eyepacs_cls']:
    !python module1/dataprep/make_eyepacs_classification_pkl.py --eyepacs-root "{EYEPACS_DIR}" --out "{EYEPACS_PKL}" --split official

In [ ]:
%cd {DRG}/dr_classification
if RUN['fgadr_cls']:
    train_cls('fgadr', 'fgadr_poc.yaml', FGADR_PKL)

In [ ]:
%cd {DRG}/dr_classification
if RUN['idrid_cls']:
    train_cls('idrid', 'idrid_poc.yaml', IDRID_PKL)

In [ ]:
# EyePACS: ~35k training images x 25 epochs at 512 px -- several hours on a Colab GPU. Resumes
# from Drive after a disconnect (like the other runs here): just re-run the notebook.
%cd {DRG}/dr_classification
if RUN['eyepacs_cls']:
    train_cls('eyepacs', 'eyepacs_poc.yaml', EYEPACS_PKL)

In [ ]:
# Item 7.1: validate the optic-disc laterality heuristic (module1/resolve_eye_laterality.py) on
# EyePACS, whose file names carry the truth (<id>_left / <id>_right): accuracy and uncertain rate per
# --margin and per OD/OS convention. Notebooks 04_05-07 resolve Tianjin's eyes with the best settings
# from this file (module1_cache_prefinals/laterality_eyepacs_validation.json on Drive).
# Independent of RUN['eyepacs_cls']: only a random 1,000 _left/_right images are pulled out of the zip.
LAT_VALIDATION = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_eyepacs_validation.json')
RUN_LATERALITY_VALIDATION = True
_lat_dir = EYEPACS_DIR if 'EYEPACS_DIR' in globals() and os.path.isdir(EYEPACS_DIR) else None
if RUN_LATERALITY_VALIDATION and _lat_dir is None:
    import random, zipfile
    _z = sorted(glob.glob(os.path.join(DRIVE_DATA_DIR, '*eyepacs*.zip')) + glob.glob(os.path.join(DRIVE_DATA_DIR, '*EyePACS*.zip')))
    if _z:
        _lat_dir = '/content/data/eyepacs_laterality_sample'
        if not glob.glob(os.path.join(_lat_dir, '**', '*.*'), recursive=True):
            with zipfile.ZipFile(_z[0]) as _zf:
                _names = [n for n in _zf.namelist()
                          if os.path.splitext(n)[0].lower().endswith(('_left', '_right'))
                          and n.lower().endswith(('.jpeg', '.jpg', '.png'))]
                for _n in random.Random(0).sample(_names, min(1000, len(_names))):
                    _zf.extract(_n, _lat_dir)
if RUN_LATERALITY_VALIDATION and _lat_dir:
    os.makedirs(os.path.dirname(LAT_VALIDATION), exist_ok=True)
    %cd {REPO_CODE_DIR}
    !python module1/resolve_eye_laterality.py --validate-eyepacs-dir "{_lat_dir}" --validate-n 1000 --validate-out "{LAT_VALIDATION}"
else:
    print('No EyePACS zip on Drive (or RUN_LATERALITY_VALIDATION = False) -- laterality validation skipped')

## Segmentation -- FGADR (DRG-Net `dr_segmentation/train_fgadr.py`)

One U-Net (ResNet50 encoder) per lesion, `--seed 765 --preprocess 2` (contrast enhancement), as in
DRG-Net's README. Every `SEG_EVAL_EVERY` epochs it evaluates AP/AUC on the 20% eval slice and saves
`model_2.pth.tar` on a new best AP; every `SEG_SAVE_EVERY` epochs it saves `model_last.pth.tar`
for resuming. Both are written to the VM disk and copied to Drive every `SEG_MIRROR_EVERY` epochs
and at the end of each lesion (`finish_seg` then checks they arrived).

In [ ]:
SEG_DRG = f'{DRG}/dr_segmentation'
SEG_OUT = os.path.join(REPL_DIR, 'segmentation')
# DRG-Net writes checkpoints to dr_segmentation/results/<run>/ (FGADR: models_FGADR_NO_TATL_<lesion>,
# IDRiD: new_models_<lesion>, DDR: ddr_models_<lesion>, combined: comb_models_<lesion>). That folder
# stays on the VM disk; the patched train scripts copy each run to SEG_OUT on Drive every
# SEG_MIRROR_EVERY epochs and at the end, and finish_seg() checks the files arrived.
SEG_LOCAL = os.path.join(SEG_DRG, 'results')
if os.path.islink(SEG_LOCAL):   # runtimes set up by an older version of this notebook linked it to Drive
    os.unlink(SEG_LOCAL)
os.makedirs(SEG_LOCAL, exist_ok=True)
os.makedirs(SEG_OUT, exist_ok=True)

def finish_seg(run_name):
    """After a lesion run (or when it is already finished): copy its files to Drive and check the
    best model (model_2.pth.tar, what the pipeline loads) and resume file are there. A run that
    finished without a best model on Drive (e.g. lost to a full Drive) falls back to its final-epoch
    weights -- printed, so it can be reported."""
    local_dir, drive_dir = os.path.join(SEG_LOCAL, run_name), os.path.join(SEG_OUT, run_name)
    if os.path.isdir(local_dir):
        sync_dir(local_dir, drive_dir)
    best, last = os.path.join(drive_dir, 'model_2.pth.tar'), os.path.join(drive_dir, 'model_last.pth.tar')
    if not os.path.isfile(best) and os.path.isfile(last):
        print(f'  ⚠ {run_name}: no best-epoch model (model_2.pth.tar) on Drive -- using the final-epoch '
              'weights (model_last.pth.tar) as model_2.pth.tar. Report this run as final-epoch weights.')
        shutil.copyfile(last, best)
    check_on_drive(drive_dir, ['model_2.pth.tar', 'model_last.pth.tar'], run_name, local_dir)

def restore_ap_log(prefix, lesion):
    """DRG-Net appends each evaluation's AP to <prefix><lesion>2.txt in its working folder, which a
    new runtime starts empty; put back the Drive copy first so a resumed run keeps the earlier
    epochs (the copy back to Drive after training would otherwise overwrite them)."""
    _logs = glob.glob(os.path.join(REPL_DIR, 'logs', f'{prefix}{lesion}*.txt'))
    if not _logs and RUN_SUFFIX:  # first run on this branch: keep the earlier epochs' AP history
        _logs = glob.glob(os.path.join(PREV_REPL_DIR, 'logs', f'{prefix}{lesion}*.txt'))
    for _log in _logs:
        _dst = os.path.join(SEG_DRG, os.path.basename(_log))
        if not os.path.isfile(_dst):
            shutil.copy(_log, _dst)

def seg_env(run_name, dataset, lesion):
    """Env vars for one lesion run: epochs/eval/save/Drive-copy intervals, plus resume state if any
    (restored from Drive to the VM disk first). None = already finished."""
    epochs = seg_epochs(dataset, lesion)
    dice = SEG_DICE_WEIGHT.get(dataset, {}).get(lesion, 0)
    env = (f"DRG_SEG_EPOCHS={epochs} DRG_SEG_DICE_WEIGHT={dice} DRG_SEG_EVAL_EVERY={SEG_EVAL_EVERY[dataset]} "
           f"DRG_SEG_SAVE_EVERY={SEG_SAVE_EVERY} DRG_SEG_MIRROR_DIR='{SEG_OUT}' "
           f"DRG_SEG_MIRROR_EVERY={SEG_MIRROR_EVERY}")
    if RESUME:
        restore_dir(os.path.join(SEG_OUT, run_name), os.path.join(SEG_LOCAL, run_name))
    last = os.path.join(SEG_LOCAL, run_name, 'model_last.pth.tar')
    if RESUME and os.path.isfile(last):
        import torch
        st = torch.load(last, map_location='cpu', weights_only=False)
        if st['epoch'] + 1 >= epochs:
            print(f'  {run_name}: already finished ({st["epoch"] + 1} epochs) -- skipping')
            return None
        print(f'  {run_name}: resuming at epoch {st["epoch"] + 2}/{epochs} (best AP so far {st["best_ap"]:.4f})')
        env += f' DRG_SEG_RESUME="{last}" DRG_SEG_BEST_AP={st["best_ap"]}'
    return env

if RUN['fgadr_seg']:
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'data', 'DR_Seg_Grading_Label_Filtered.csv'),
                os.path.join(FGADR_ROOT, 'DR_Seg_Grading_Label_Filtered.csv'))
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'configs', 'config_fgadr_seg_poc.py'),
                os.path.join(SEG_DRG, 'config_fgadr.py'))
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        print(f'=== FGADR {lesion}: {seg_epochs("fgadr", lesion)} epochs ===')
        _run = f'models_FGADR_NO_TATL_{lesion.lower()}'
        _env = seg_env(_run, 'fgadr', lesion)
        if _env is None:
            finish_seg(_run)
            continue
        restore_ap_log('ap_during_learning_FGADR_NO_TATL_', lesion)
        !{_env} python -u train_fgadr.py --seed 765 --preprocess 2 --lesion {lesion} 2>&1 | python -u {REPO_CODE_DIR}/module1/seg_progress.py --every 10 --log "{REPL_DIR}/logs/seg_fgadr_{lesion}.log"
        for _log in glob.glob(f'{SEG_DRG}/ap_during_learning_FGADR_NO_TATL_{lesion}*.txt'):
            shutil.copy(_log, os.path.join(REPL_DIR, 'logs'))
        finish_seg(_run)

## Segmentation -- IDRiD (DRG-Net `dr_segmentation/train.py`)

`prepare_idrid_segmentation.py` lays IDRiD's "A. Segmentation" set out the way DRG-Net's
`utils.get_images()` reads it, and generates the field-of-view masks DRG-Net needs (IDRiD doesn't
ship them; checked against DRG-Net's bundled IDRiD samples). Then `train.py` per lesion (eval every
`SEG_EVAL_EVERY` epochs on the 20% eval slice, resumable like FGADR), and DRG-Net's
`evaluate_model.py` on the official 27-image test set.

In [ ]:
IDRID_SEG_DIR = '/content/data/IDRiD_seg_drgnet'
# Laid out even when IDRiD isn't trained this session: the evaluation scores every model on IDRiD's test set.
if not os.path.isdir(os.path.join(IDRID_SEG_DIR, 'OriginalImages', 'TestingSet')):
    %cd {REPO_CODE_DIR}
    !python module1/dataprep/prepare_idrid_segmentation.py --idrid-root "{IDRID_ROOT}" --out "{IDRID_SEG_DIR}"
if RUN['idrid_seg']:
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'configs', 'config_idrid_seg.py'),
                os.path.join(SEG_DRG, 'config.py'))
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        print(f'=== IDRiD {lesion}: {seg_epochs("idrid", lesion)} epochs ===')
        _run = f'new_models_{lesion.lower()}'
        _env = seg_env(_run, 'idrid', lesion)
        if _env is None:
            finish_seg(_run)
            continue
        restore_ap_log('new_ap_during_learning_', lesion)
        !{_env} python -u train.py --seed 765 --preprocess 2 --lesion {lesion} 2>&1 | python -u {REPO_CODE_DIR}/module1/seg_progress.py --every 10 --log "{REPL_DIR}/logs/seg_idrid_{lesion}.log"
        for _log in glob.glob(f'{SEG_DRG}/new_ap_during_learning_{lesion}*.txt'):
            shutil.copy(_log, os.path.join(REPL_DIR, 'logs'))
        finish_seg(_run)

## Segmentation -- DDR (our addition; DRG-Net's IDRiD recipe, `dr_segmentation/train.py`)

`prepare_ddr_segmentation.py` converts DDR's `lesion_segmentation/` set (383 train / 149 val /
225 test, per-lesion masks) into the IDRiD layout: every image cropped to the retina, scaled and
padded to `DDR_SIZE` x `DDR_SIZE` (DRG-Net evaluates whole images in batches, so they must share
one size, and keeps them all in RAM), masks binarised, field-of-view masks generated. If you change
`DDR_SIZE`, delete `/content/data/DDR_seg_drgnet` (or restart the runtime) so it is rebuilt. `train.py` then trains on the 383 train images and picks the best epoch on the 149 val
images (`DRG_SEG_TRAIN_RATIO` from `ddr_split.json`); the 225 test images are only used by the
evaluation below. Checkpoints go to `segmentation/ddr_models_<lesion>/` (separate from IDRiD's
`new_models_<lesion>/`), resumable like the others.

In [ ]:
DDR_SEG_DIR = '/content/data/DDR_seg_drgnet'
if (RUN['ddr_seg'] or RUN['comb_seg']) and not os.path.isfile(os.path.join(DDR_SEG_DIR, 'ddr_split.json')):
    %cd {REPO_CODE_DIR}
    !python module1/dataprep/prepare_ddr_segmentation.py --ddr-root "{DDR_ROOT}" --out "{DDR_SEG_DIR}" --size {DDR_SIZE}
if RUN['ddr_seg']:
    DDR_SPLIT = json.load(open(os.path.join(DDR_SEG_DIR, 'ddr_split.json')))
    print('DDR split:', DDR_SPLIT)
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'configs', 'config_idrid_seg.py'),
                os.path.join(SEG_DRG, 'config.py'))
    DDR_ENV = (f'DRG_SEG_IMAGE_DIR="{DDR_SEG_DIR}/" DRG_SEG_RUN_PREFIX=ddr_ '
               f'DRG_SEG_TRAIN_RATIO={DDR_SPLIT["train_ratio"]}')
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        print(f'=== DDR {lesion}: {seg_epochs("ddr", lesion)} epochs ===')
        _run = f'ddr_models_{lesion.lower()}'
        _env = seg_env(_run, 'ddr', lesion)
        if _env is None:
            finish_seg(_run)
            continue
        restore_ap_log('ddr_ap_during_learning_', lesion)
        !{DDR_ENV} {_env} python -u train.py --seed 765 --preprocess 2 --lesion {lesion} 2>&1 | python -u {REPO_CODE_DIR}/module1/seg_progress.py --every 10 --log "{REPL_DIR}/logs/seg_ddr_{lesion}.log"
        for _log in glob.glob(f'{SEG_DRG}/ddr_ap_during_learning_{lesion}*.txt'):
            shutil.copy(_log, os.path.join(REPL_DIR, 'logs'))
        finish_seg(_run)

## Segmentation -- FGADR + DDR combined (the pipeline's lesion models)

`prepare_combined_segmentation.py` pools FGADR's training rows (DRG-Net's split: first 60% of
`DR_Seg_Grading_Label_Filtered.csv`) with DDR's 383 train images, and FGADR's eval rows (next 20%)
with DDR's 149 val images, all at `DDR_SIZE` x `DDR_SIZE`. `train.py` trains one model per lesion
on the pooled training images and picks the best epoch on the pooled eval images. The FGADR and
DDR test images are not pooled -- the evaluation below scores this model on each test set
separately. Checkpoints: `segmentation/comb_models_<lesion>/model_2.pth.tar`, resumable like the
others. RAM: ~850 images are held in memory (masks 1-channel); if the runtime crashes, use a
High-RAM runtime.

In [ ]:
# Needs the cells above in this runtime (settings, unzip, clone): use Runtime -> Run all.
for _v in ('RUN', 'REPO_CODE_DIR', 'FGADR_ROOT', 'DDR_ROOT', 'DDR_SIZE', 'SEG_DRG', 'seg_env'):
    assert _v in globals(), f'{_v} is not defined -- run the notebook from the top (Runtime -> Run all), not from this cell'
DDR_SEG_DIR = '/content/data/DDR_seg_drgnet'   # same as the DDR cell, so this cell doesn't depend on it
COMB_SEG_DIR = '/content/data/COMB_seg_drgnet'
if RUN['comb_seg']:
    %cd {REPO_CODE_DIR}
    if not os.path.isfile(os.path.join(DDR_SEG_DIR, 'ddr_split.json')):
        !python module1/dataprep/prepare_ddr_segmentation.py --ddr-root "{DDR_ROOT}" --out "{DDR_SEG_DIR}" --size {DDR_SIZE}
    assert os.path.isfile(os.path.join(DDR_SEG_DIR, 'ddr_split.json')), (
        'DDR preparation failed -- see its output above (is the DDR zip on Drive, with a lesion_segmentation folder inside?)')
    if not os.path.isfile(os.path.join(COMB_SEG_DIR, 'combined_split.json')):
        !python module1/dataprep/prepare_combined_segmentation.py --fgadr-root "{FGADR_ROOT}" --filtered-csv "{REPO_CODE_DIR}/module1/data/DR_Seg_Grading_Label_Filtered.csv" --ddr-seg-dir "{DDR_SEG_DIR}" --out "{COMB_SEG_DIR}"
    assert os.path.isfile(os.path.join(COMB_SEG_DIR, 'combined_split.json')), (
        'Combining FGADR + DDR failed -- see the output above')
    COMB_SPLIT = json.load(open(os.path.join(COMB_SEG_DIR, 'combined_split.json')))
    print('Combined split:', COMB_SPLIT)
    shutil.copy(os.path.join(REPO_CODE_DIR, 'module1', 'configs', 'config_idrid_seg.py'),
                os.path.join(SEG_DRG, 'config.py'))
    COMB_ENV = (f'DRG_SEG_IMAGE_DIR="{COMB_SEG_DIR}/" DRG_SEG_RUN_PREFIX=comb_ '
                f'DRG_SEG_TRAIN_RATIO={COMB_SPLIT["train_ratio"]}')
    %cd {SEG_DRG}
    for lesion in SEG_LESIONS:
        print(f'=== FGADR + DDR {lesion}: {seg_epochs("comb", lesion)} epochs ===')
        _run = f'comb_models_{lesion.lower()}'
        _env = seg_env(_run, 'comb', lesion)
        if _env is None:
            finish_seg(_run)
            continue
        restore_ap_log('comb_ap_during_learning_', lesion)
        !{COMB_ENV} {_env} python -u train.py --seed 765 --preprocess 2 --lesion {lesion} 2>&1 | python -u {REPO_CODE_DIR}/module1/seg_progress.py --every 10 --log "{REPL_DIR}/logs/seg_comb_{lesion}.log"
        for _log in glob.glob(f'{SEG_DRG}/comb_ap_during_learning_{lesion}*.txt'):
            shutil.copy(_log, os.path.join(REPL_DIR, 'logs'))
        finish_seg(_run)

In [ ]:
# Test-set evaluation: every trained model on every available test set (a model is scored
# whenever its checkpoint exists, whether or not it was trained this session).
#  FGADR test: our evaluate_segmentation_dice_iou.py -- AP (AUC-PR) / ROC-AUC on the raw soft
#         predictions (plus Dice/IoU), on the held-out last 20% (80 images). DRG-Net's own 'test'
#         phase returns ALL 401 images incl. the training ones, so we use the held-out slice.
#         SEG_EVAL_MODE 'tiled' (default) = the pipeline's inference (retina crop, ~1280 px retina,
#         overlapping 512 px tiles -- module1/apply_to_progression_data.py); 'resize' = the earlier
#         numbers (whole image resized to 1280 x 1280, no crop).
#  IDRiD / DDR test: DRG-Net's own evaluate_model.py on the official IDRiD (27) / DDR (225) test
#         sets. NB its printout says "AUC:" but the number is average precision.
# Compare models within a column (same test set + same script); the IDRiD column is the
# fairest check of whether pooling FGADR + DDR helped, since neither trains on IDRiD.
SEG_MODELS = {'fgadr': 'models_FGADR_NO_TATL_', 'idrid': 'new_models_', 'ddr': 'ddr_models_',
              'comb': 'comb_models_'}
SEG_EVAL_MODE = 'tiled'
SEG_EVAL_ARGS = '' if SEG_EVAL_MODE == 'tiled' else ' --seg-mode resize --no-retina-crop --image-size 1280'

def seg_ckpt(prefix, lesion):
    """This branch's model_2.pth.tar, else the earlier run's (models not re-trained this session are
    still scored -- with this branch's evaluation)."""
    for _root in (SEG_OUT, os.path.join(PREV_REPL_DIR, 'segmentation')):
        _ck = os.path.join(_root, f'{prefix}{lesion.lower()}', 'model_2.pth.tar')
        if os.path.isfile(_ck):
            return _ck
    return None
_csv = f'{REPO_CODE_DIR}/module1/data/DR_Seg_Grading_Label_Filtered.csv'
_tests = {'fgadr': os.path.isdir(os.path.join(FGADR_ROOT, 'Original_Images')),
          'idrid': os.path.isdir(os.path.join(IDRID_SEG_DIR, 'OriginalImages', 'TestingSet')),
          'ddr': os.path.isdir(os.path.join(DDR_SEG_DIR, 'OriginalImages', 'TestingSet'))}
results = {'seg_test_AP': {}, 'seg_eval_mode': SEG_EVAL_MODE}

def _drg_test_ap(ck, lesion, image_dir):
    _o = !DRG_SEG_IMAGE_DIR="{image_dir}/" python evaluate_model.py --seed 765 --preprocess 2 --lesion {lesion} --model "{ck}"
    _ap = [l for l in _o if l.startswith('AUC:')]
    if not _ap:
        print('\n'.join(_o[-5:]))
    return float(_ap[-1].split()[-1]) if _ap else None

for model, prefix in SEG_MODELS.items():
    for lesion in SEG_LESIONS:
        ck = seg_ckpt(prefix, lesion)
        if ck is None:
            continue
        row = results['seg_test_AP'].setdefault(model, {})
        results.setdefault('seg_checkpoints', {})[f'{model}_{lesion}'] = ck
        if _tests['fgadr']:
            %cd {REPO_CODE_DIR}/module1
            _curve = os.path.join(REPL_DIR, f'roc_curve_{lesion}.json' if model == 'fgadr' else f'roc_curve_{model}_{lesion}.json')
            _o = !python evaluate_segmentation_dice_iou.py --fgadr-root "{FGADR_ROOT}" --filtered-csv "{_csv}" --checkpoint "{ck}" --lesion {lesion}{SEG_EVAL_ARGS} --curve-out "{_curve}"
            _m = {k: float(re.search(rf'{k}:\s+([0-9.]+)', '\n'.join(_o)).group(1))
                  for k in ('Mean Dice', 'Mean IoU', 'Mean AUC-PR', 'Mean AUC-ROC')
                  if re.search(rf'{k}:\s+([0-9.]+)', '\n'.join(_o))}
            if not _m:
                print(f'{model} {lesion} on FGADR test -- no score:', '\n'.join(_o[-3:]))
            row.setdefault('fgadr_test', {})[lesion] = _m.get('Mean AUC-PR')
            results[f'{model}_model_on_fgadr_test_{lesion}'] = _m
        %cd {SEG_DRG}
        if _tests['idrid']:
            row.setdefault('idrid_test', {})[lesion] = _drg_test_ap(ck, lesion, IDRID_SEG_DIR)
        if _tests['ddr']:
            row.setdefault('ddr_test', {})[lesion] = _drg_test_ap(ck, lesion, DDR_SEG_DIR)

import pandas as pd
_rows = [{'model': m, 'test set': t, **v} for m, d in results['seg_test_AP'].items() for t, v in d.items()]
if _rows:
    print('Lesion segmentation AP by model (rows) and test set -- compare models within the same test set:')
    display(pd.DataFrame(_rows).set_index(['test set', 'model']).sort_index().round(4))

In [ ]:
# Figures for every trained model (the pipeline's combined model first), saved to Drive:
#  * ROC curves per lesion, and image | ground truth | prediction examples -- on FGADR's held-out
#    test images (the ROC data comes from the evaluation cell above, so run that first);
#  * validation AP vs. epoch per lesion (the best epoch, kept as model_2.pth.tar, is starred);
#  * validation kappa/accuracy vs. epoch for each trained grader;
#  * one FGADR sample with its lesion masks + Lesion Burden Score (LBS).
from IPython.display import Image as IPImage, display
FIG_DIR = os.path.join(REPL_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
%cd {REPO_CODE_DIR}/module1
_csv = f'{REPO_CODE_DIR}/module1/data/DR_Seg_Grading_Label_Filtered.csv'
SEG_FIG_MODELS = [  # key, label, checkpoint folder prefix, AP log prefix (in logs/)
    ('comb', 'FGADR + DDR combined (pipeline)', 'comb_models_', 'comb_ap_during_learning_'),
    ('fgadr', 'FGADR only', 'models_FGADR_NO_TATL_', 'ap_during_learning_FGADR_NO_TATL_'),
    ('ddr', 'DDR only', 'ddr_models_', 'ddr_ap_during_learning_'),
    ('idrid', 'IDRiD', 'new_models_', 'new_ap_during_learning_'),
]
_made = []
for key, label, ck_prefix, log_prefix in SEG_FIG_MODELS:
    _cks = {l: seg_ckpt(ck_prefix, l) or '' for l in SEG_LESIONS}  # seg_ckpt: evaluation cell above
    if not any(os.path.isfile(c) for c in _cks.values()):
        continue
    _name = '' if key == 'fgadr' else f'_{key}'  # FGADR-only keeps the original file names
    _curves = ' '.join(f'--curve {l}="{REPL_DIR}/roc_curve{_name}_{l}.json"' for l in SEG_LESIONS
                       if os.path.isfile(f'{REPL_DIR}/roc_curve{_name}_{l}.json'))
    if _curves:
        !python plot_figures.py roc {_curves} --title "Lesion segmentation ROC on FGADR test -- {label}" --out "{FIG_DIR}/module1_roc_curves{_name}.png"
    _logs = ' '.join(f'--log {l}="{REPL_DIR}/logs/{log_prefix}{l}2.txt"' for l in SEG_LESIONS
                     if os.path.isfile(f'{REPL_DIR}/logs/{log_prefix}{l}2.txt'))
    if _logs:
        !python plot_figures.py seg-learning {_logs} --title "{label}" --out "{FIG_DIR}/module1_seg_learning{_name}.png"
    if key in ('comb', 'fgadr'):  # qualitative examples for the pipeline model and its FGADR-only baseline
        for lesion, ck in _cks.items():
            if os.path.isfile(ck):
                !python plot_figures.py seg-grid --fgadr-root "{FGADR_ROOT}" --filtered-csv "{_csv}" --checkpoint "{ck}" --lesion {lesion} --n-examples 3 --image-size 1280 --title-suffix " -- {label}" --out "{FIG_DIR}/module1_seg_grid{_name}_{lesion}.png"
    _made.append(label)
print('Segmentation figures made for:', _made or 'no trained segmentation model yet')
for name in ('fgadr', 'idrid', 'eyepacs'):
    _log = os.path.join(REPL_DIR, 'logs', f'cls_{name}.log')
    if os.path.isfile(_log):
        _label = 'EyePACS' if name == 'eyepacs' else name.upper()  # not inline: a Python conditional inside !python is a bash syntax error
        !python plot_figures.py cls-curve --log "{_log}" --dataset-name "{_label}" --out "{FIG_DIR}/module1_cls_curve_{name}.png"
import pandas as pd
_sample = pd.read_csv(_csv)['Image_Name'].iloc[0]
!python compute_lbs.py --fgadr-root "{FGADR_ROOT}" --image-name {_sample} --out "{FIG_DIR}/module1_lbs_sample.png"
for _f in sorted(glob.glob(f'{FIG_DIR}/*.png'), key=lambda f: ('_comb' not in f, f)):
    print(_f); display(IPImage(_f))

In [ ]:
# Summary table: classification acc/kappa (best-validation and final checkpoints) from the
# tee'd logs, segmentation AP/AUC from the eval outputs above. Saved to Drive as JSON.
def parse_cls(log):
    if not os.path.isfile(log):
        return None
    t = open(log).read()
    # last two = this run's best-validation and final checkpoints (logs are appended across resumes)
    accs = re.findall(r'test acc: ([0-9.]+)', t)[-2:]
    kappas = re.findall(r'quadratic kappa: (-?[0-9.]+)', t)[-2:]
    return {'best_validation': {'acc': accs[0] if accs else None, 'kappa': kappas[0] if kappas else None},
            'final': {'acc': accs[1] if len(accs) > 1 else None, 'kappa': kappas[1] if len(kappas) > 1 else None}}

for name in ('fgadr', 'idrid', 'eyepacs'):
    results[f'cls_{name}'] = parse_cls(os.path.join(REPL_DIR, 'logs', f'cls_{name}.log'))
json.dump(results, open(os.path.join(REPL_DIR, 'replication_results.json'), 'w'), indent=2)
print(json.dumps(results, indent=2))

## Using these checkpoints in the pipeline

Notebooks 04_05 and 06-10 have a `MODULE1_SOURCE` setting in their first code cell:

| `MODULE1_SOURCE` | Classifier | Lesion segmenters | Outputs go to (suffix) |
|---|---|---|---|
| `'drgnet_combined'` (default) | FGADR (this notebook) | **FGADR + DDR combined** (`comb_models_*`) | `_drgnet_comb` |
| `'drgnet'` | FGADR (this notebook) | FGADR only (`models_FGADR_NO_TATL_*`) | `_drgnet` |
| `'poc'` | notebooks 02/03 | notebooks 02/03 | no suffix |

Each setting writes to its own folders (`module1_cache_drgnet_comb/`, `module2_runs/..._drgnet_comb`,
...), so switching never overwrites another setting's caches or models. Run order with the
DRG-Net models: **11 -> 04_05 -> 09 -> 06 -> 07 -> 08 -> 10**.

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')